# PFA    Classification des Maladies Pulmonaires
**NIH Chest X-ray Dataset 15 Pathologies**

---
## CELLULE 1    Vérification GPU + chemins

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score,
                              confusion_matrix, classification_report)
warnings.filterwarnings('ignore')

In [ ]:
# Chemins Kaggle
DATA_PATH = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'
WORK_PATH = '/kaggle/working'

os.makedirs(f'{WORK_PATH}/models', exist_ok=True)
os.makedirs(f'{WORK_PATH}/logs',   exist_ok=True)

# Hyperparametres globaux
IMG_SIZE   = 224
BATCH_SIZE = 32
ALL_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion',    'Emphysema',    'Fibrosis',       'Hernia',
    'Infiltration','Mass',         'No Finding',     'Nodule',
    'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
N_CLASSES = len(ALL_LABELS)

#   Verification GPU  
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'GPU detecte : {gpus[0].name}')
else:
    print('Aucun GPU    active-le : Settings → Accelerator → GPU P100')

#   Verification dataset  
csv_path = os.path.join(DATA_PATH, 'Data_Entry_2017.csv')
if os.path.exists(csv_path):
    print(f'Dataset NIH disponible : {DATA_PATH}')
else:
    print(f'ERREUR : CSV introuvable dans {DATA_PATH}')
    print('Verifie que tu es bien sur un Kaggle Notebook cree depuis le dataset NIH')

#   Liste des fichiers disponibles  
print('\nContenu du dataset :')
for f in sorted(os.listdir(DATA_PATH))[:15]:
    print(f'  {f}')

print(f'\nTensorFlow : {tf.__version__}')

In [ ]:
# =============================================================
# CELLULE 2    Chargement données + Prétraitement complet
# =============================================================

# Chargement CSV et indexation des chemins  
df = pd.read_csv(os.path.join(DATA_PATH, 'Data_Entry_2017.csv'))
print(f'CSV : {len(df):,} images')

all_paths = glob.glob(os.path.join(DATA_PATH, 'images_*/images/*.png'))
path_dict = {os.path.basename(p): p for p in all_paths}
df['FilePath'] = df['Image Index'].map(path_dict)
df = df.dropna(subset=['FilePath']).reset_index(drop=True)
print(f'Images indexees : {len(df):,}')

# Encodage multi-label
label_dummies = df['Finding Labels'].str.get_dummies(sep='|')
for label in ALL_LABELS:
    if label not in label_dummies.columns:
        label_dummies[label] = 0
label_dummies = label_dummies[ALL_LABELS]
df = pd.concat([df, label_dummies], axis=1)

# Split dataset 
with open(os.path.join(DATA_PATH, 'train_val_list.txt')) as f:
    tv_list = set(f.read().splitlines())
with open(os.path.join(DATA_PATH, 'test_list.txt')) as f:
    te_list = set(f.read().splitlines())

df_trainval = df[df['Image Index'].isin(tv_list)].copy()
df_test     = df[df['Image Index'].isin(te_list)].copy()
df_train, df_val = train_test_split(
    df_trainval, test_size=0.2, random_state=42,
    stratify=df_trainval['No Finding']
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# exemple sous-ensemble equilibre
sampled = []
for label in ALL_LABELS:
    limit  = 7000 if label == 'No Finding' else 4000
    subset = df_train[df_train[label] == 1].head(limit)
    sampled.append(subset)
df_train_s = pd.concat(sampled).drop_duplicates(subset='Image Index')
df_train_s = df_train_s.sample(frac=1, random_state=42).reset_index(drop=True)

# class weights
class_weights = {}
n = len(df_train_s)
for i, label in enumerate(ALL_LABELS):
    n_pos = df_train_s[label].sum()
    class_weights[i] = n / (N_CLASSES * n_pos) if n_pos > 0 else 1.0

# distribution des classes  
counts = label_dummies.sum().sort_values(ascending=False)
plt.figure(figsize=(13, 4))
colors = ['#FF5722' if c == 'No Finding' else '#2196F3' for c in counts.index]
plt.bar(counts.index, counts.values, color=colors)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.title('Distribution des pathologies    NIH Chest X-ray')
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/distribution.png', dpi=100)
plt.show()

print(f'Train : {len(df_train_s):,} | Val : {len(df_val):,} | Test : {len(df_test):,}')
print(f'Data leakage train/test : '
      f'{len(set(df_train["Image Index"]) & set(df_test["Image Index"]))} (doit etre 0)')



In [ ]:
# =============================================================
# CELLULE A    Split train/val/test
# =============================================================

from sklearn.model_selection import train_test_split

# Split du dataset
with open(os.path.join(DATA_PATH, 'train_val_list.txt')) as f:
    tv_list = set(f.read().splitlines())
with open(os.path.join(DATA_PATH, 'test_list.txt')) as f:
    te_list = set(f.read().splitlines())

df_trainval = df[df['Image Index'].isin(tv_list)].copy()
df_test     = df[df['Image Index'].isin(te_list)].copy()
df_train, df_val = train_test_split(
    df_trainval, test_size=0.2, random_state=42,
    stratify=df_trainval['No Finding']
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

total = len(df_train) + len(df_val) + len(df_test)
pct_train = len(df_train) / total * 100
pct_val   = len(df_val)   / total * 100
pct_test  = len(df_test)  / total * 100

print('SPLIT DU DATASET    NIH Chest X-ray')
print('='*50)
print(f'  Total images     : {total:,}')
print(f'  Train            : {len(df_train):,}  ({pct_train:.1f}%)')
print(f'  Validation       : {len(df_val):,}  ({pct_val:.1f}%)')
print(f'  Test             : {len(df_test):,}  ({pct_test:.1f}%)')
print(f'  Data leakage     : {len(set(df_train["Image Index"]) & set(df_test["Image Index"]))} (doit etre 0)')

# Visualisation du split
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# pie chart
sizes  = [len(df_train), len(df_val), len(df_test)]
labels = [f'Train\n{len(df_train):,}\n({pct_train:.1f}%)',
          f'Validation\n{len(df_val):,}\n({pct_val:.1f}%)',
          f'Test\n{len(df_test):,}\n({pct_test:.1f}%)']
colors = ['#2196F3', '#FF9800', '#4CAF50']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Repartition train / validation / test', fontweight='bold')

# bar chart
axes[1].bar(['Train', 'Validation', 'Test'], sizes, color=colors, edgecolor='white')
for i, (v, p) in enumerate(zip(sizes, [pct_train, pct_val, pct_test])):
    axes[1].text(i, v + 200, f'{v:,}\n({p:.1f}%)', ha='center', fontsize=9)
axes[1].set_ylabel('Nombre images')
axes[1].set_title('Nombre images par split', fontweight='bold')
axes[1].set_ylim([0, max(sizes) * 1.15])

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/split_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique split sauvegarde')

In [ ]:
# =============================================================
# CELLULE B    Distribution des classes
# =============================================================

# Encodage multi-label
label_dummies = df['Finding Labels'].str.get_dummies(sep='|')
for label in ALL_LABELS:
    if label not in label_dummies.columns:
        label_dummies[label] = 0
label_dummies = label_dummies[ALL_LABELS]
df = pd.concat([df, label_dummies], axis=1)

for split_df in [df_train, df_val, df_test]:
    for label in ALL_LABELS:
        if label not in split_df.columns:
            lbl = split_df['Finding Labels'].str.get_dummies(sep='|')
            for l in ALL_LABELS:
                if l not in lbl.columns:
                    lbl[l] = 0
            for l in ALL_LABELS:
                split_df[l] = lbl[l].values
            break

counts = label_dummies.sum().sort_values(ascending=False)
total  = len(df)

# affichage
print('DISTRIBUTION DES CLASSES    NIH Chest X-ray')
print('='*55)
print(f'{"Pathologie":<25} {"Nb images":>10}  {"Pourcentage":>12}')
print('-'*55)
for label, count in counts.items():
    bar = '|' * int(count / total * 50)
    print(f'{label:<25} {count:>10,}  {count/total*100:>10.2f}%  {bar}')
print('-'*55)
print(f'{"TOTAL":<25} {total:>10,}  {100:>10.1f}%')

# Graphique de distribution
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Distribution des pathologies    NIH Chest X-ray Dataset',
             fontsize=13, fontweight='bold')

# barplot
colors = ['#FF5722' if c == 'No Finding' else '#2196F3' for c in counts.index]
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{count/total*100:.1f}%', ha='center', va='bottom', fontsize=8)
axes[0].set_ylabel('Nombre images'); axes[0].set_xticks(range(len(counts)))
axes[0].set_xticklabels(counts.index, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Nombre images par pathologie (rouge = classe dominante)', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# distribution par split
train_counts = df_train[ALL_LABELS].sum()
val_counts   = df_val[ALL_LABELS].sum()
test_counts  = df_test[ALL_LABELS].sum()
x = range(len(ALL_LABELS))
w = 0.28
axes[1].bar([i-w for i in x], train_counts, width=w, label='Train', color='#2196F3', alpha=0.8)
axes[1].bar(x,               val_counts,   width=w, label='Validation', color='#FF9800', alpha=0.8)
axes[1].bar([i+w for i in x], test_counts,  width=w, label='Test', color='#4CAF50', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(ALL_LABELS, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Nombre images')
axes[1].set_title('Distribution par split (train / validation / test)', fontsize=11)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Distribution sauvegardee')

In [ ]:
# =============================================================
# CELLULE C    Images originales du dataset
# =============================================================
print('EXEMPLES DU DATASET ORIGINAL    avant augmentation')
print('='*55)

# supprimer les doublons
df_display = df.reset_index(drop=True)

selected_labels = ALL_LABELS[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Images originales    1 exemple par pathologie',
             fontsize=14, fontweight='bold')

for ax, label in zip(axes.flatten(), selected_labels):
    mask    = df_display['Finding Labels'].str.contains(label, na=False)
    samples = df_display[mask]
    if len(samples) == 0:
        ax.axis('off')
        continue
    img_path = samples.iloc[0]['FilePath']
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        ax.axis('off')
        continue
    ax.imshow(img, cmap='gray')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.axis('off')
    h, w = img.shape
    ax.text(0.5, -0.05, f'{w}x{h} px', transform=ax.transAxes,
            ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/images_originales.png', dpi=150, bbox_inches='tight')
plt.show()

# Infos sur les imagess
sample_path = df_display.iloc[0]['FilePath']
sample_img  = cv2.imread(sample_path)
h, w, c     = sample_img.shape
print(f'Caracteristiques des images originales :')
print(f'  Taille originale    : {w} x {h} pixels')
print(f'  Taille apres resize : 224 x 224 pixels')
print(f'  Canaux              : {c} (RGB)')
print(f'  Format              : PNG')
print(f'  Nombre total images : {len(df_display):,}')

In [ ]:
# =============================================================
# CELLULE D    Avant/Apres augmentation + calul nbre
# =============================================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator

df_display = df.reset_index(drop=True)

demo_samples = df_display[df_display['Finding Labels'].str.contains('Pneumonia', na=False)]
if len(demo_samples) == 0:
    demo_samples = df_display[df_display['Finding Labels'].str.contains('Effusion', na=False)]
demo_path = demo_samples.iloc[0]['FilePath']

# charger l'image
img_orig = cv2.cvtColor(cv2.imread(demo_path), cv2.COLOR_BGR2RGB)
img_orig = cv2.resize(img_orig, (224, 224))

# generateurs avecc differentes augmentations
augmentations = [
    ('Original',             ImageDataGenerator(rescale=1./255)),
    ('Rotation +10',         ImageDataGenerator(rescale=1./255, rotation_range=10)),
    ('Flip horizontal',      ImageDataGenerator(rescale=1./255, horizontal_flip=True)),
    ('Zoom x0.9',            ImageDataGenerator(rescale=1./255, zoom_range=0.1)),
    ('Decalage horizontal',  ImageDataGenerator(rescale=1./255, width_shift_range=0.1)),
    ('Decalage vertical',    ImageDataGenerator(rescale=1./255, height_shift_range=0.1)),
    ('Toutes augmentations', ImageDataGenerator(
        rescale=1./255, rotation_range=10, horizontal_flip=True,
        zoom_range=0.1, width_shift_range=0.05, height_shift_range=0.05)),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Avant et apres augmentation    meme image radio',
             fontsize=13, fontweight='bold')

img_tensor = np.expand_dims(img_orig, 0)
for i, (ax, (name, gen)) in enumerate(zip(axes.flatten(), augmentations)):
    it  = gen.flow(img_tensor, batch_size=1, seed=i*7)
    aug = next(it)[0]
    ax.imshow(aug)
    ax.set_title(name, fontsize=10, fontweight='bold' if name == 'Original' else 'normal')
    ax.axis('off')
    if name == 'Original':
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color('#2196F3')
            spine.set_linewidth(3)

axes.flatten()[-1].axis('off')
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()

# calcul nbre
print('NOMBRE D IMAGES AVANT ET APRES AUGMENTATION')
print('='*55)
print(f'  Images originales (train)  : {len(df_train_s):,}')
print(f'  Images originales (val)    : {len(df_val):,}')
print(f'  Images originales (test)   : {len(df_test):,}')
N_EPOCHS_TOTAL = 40
imgs_vues = len(df_train_s) * N_EPOCHS_TOTAL
print(f'\n  Transformations            : rotation, flip, zoom, decalage')
print(f'  Images vues apres {N_EPOCHS_TOTAL} epochs : ~{imgs_vues:,}')
print(f'  (augmentation a la volee    pas de stockage supplementaire)')
print(f'\n  Taille avant resize        : ~1024 x 1024 px')
print(f'  Taille apres resize        : 224 x 224 px')
print(f'  Normalisation              : pixel / 255.0 → valeurs entre 0 et 1')

In [ ]:
# =============================================================
# CELLULE 3    DataGenerators
# =============================================================

# Train augmentation 
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    horizontal_flip=True,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)
# Val/Test : normalisation
val_test_datagen = ImageDataGenerator(rescale=1./255)

def make_gen(datagen, df, shuffle):
    return datagen.flow_from_dataframe(
        dataframe=df, x_col='FilePath', y_col=ALL_LABELS,
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode='raw', color_mode='rgb', shuffle=shuffle, seed=42
    )

train_gen = make_gen(train_datagen,    df_train_s, shuffle=True)
val_gen   = make_gen(val_test_datagen, df_val,     shuffle=False)
test_gen  = make_gen(val_test_datagen, df_test,    shuffle=False)

imgs, labs = next(train_gen)
print(f'Generateurs OK')
print(f'  Batch images : {imgs.shape}  (attendu 32,224,224,3)')
print(f'  Batch labels : {labs.shape}  (attendu 32,15)')
print(f'  Steps train  : {len(train_gen)} | Steps val : {len(val_gen)}')


In [ ]:
# ============================================================
# CELLULE 4    Architecture
# ============================================================
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import DenseNet121
import tensorflow as tf

class ChannelMeanPool(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)
    def get_config(self):
        return super().get_config()

class ChannelMaxPool(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)
    def get_config(self):
        return super().get_config()

# Bloc Channel Attention  
def channel_attention(x, ratio=8):
    channels = x.shape[-1]
    avg_pool = layers.GlobalAveragePooling2D()(x)
    max_pool = layers.GlobalMaxPooling2D()(x)
    shared1  = layers.Dense(channels // ratio, activation='relu')
    shared2  = layers.Dense(channels)
    avg_out  = shared2(shared1(avg_pool))
    max_out  = shared2(shared1(max_pool))
    scale    = layers.Activation('sigmoid')(layers.Add()([avg_out, max_out]))
    scale    = layers.Reshape((1, 1, channels))(scale)
    return layers.Multiply()([x, scale])

# Bloc Spatial Attention  
def spatial_attention(x):
    avg_pool = ChannelMeanPool()(x)
    max_pool = ChannelMaxPool()(x)
    concat   = layers.Concatenate()([avg_pool, max_pool])
    scale    = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)
    return layers.Multiply()([x, scale])

# CBAM = Channel + Spatial Attention  
def cbam_block(x):
    x = channel_attention(x)
    x = spatial_attention(x)
    return x

# Bloc Résiduel tête  
def residual_dense_block(x, units=256):
    out = layers.Dense(units)(x)
    out = layers.BatchNormalization()(out)
    out = layers.Activation('relu')(out)
    out = layers.Dropout(0.3)(out)
    out = layers.Dense(units)(out)
    out = layers.BatchNormalization()(out)
    if x.shape[-1] != units:
        x = layers.Dense(units)(x)
    out = layers.Add()([out, x])
    return layers.Activation('relu')(out)

# Model final  
def build_model_v2():
    base = DenseNet121(weights='imagenet', include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False

    # multi-Scale 3 niveaux de features
    feat_mid  = base.get_layer('pool3_pool').output
    feat_deep = base.get_layer('pool4_pool').output
    feat_out  = base.output

    # CBAM sur la sortie finale
    feat_out_att = cbam_block(feat_out)

    # GlobalAveragePooling
    gap_mid  = layers.GlobalAveragePooling2D(name='gap_mid')(feat_mid)
    gap_deep = layers.GlobalAveragePooling2D(name='gap_deep')(feat_deep)
    gap_out  = layers.GlobalAveragePooling2D(name='gap_out')(feat_out_att)

    # Fusion des 3 niveaux
    fused = layers.Concatenate(name='multi_scale_fusion')([gap_mid, gap_deep, gap_out])

    # Réduction + blocs résiduels
    x = layers.Dense(1024, name='projection')(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)
    x = residual_dense_block(x, units=512)
    x = residual_dense_block(x, units=256)

    outputs = layers.Dense(N_CLASSES, activation='sigmoid', name='predictions')(x)
    return Model(inputs=base.input, outputs=outputs, name='DenseNet121_CBAM_MultiScale')

# construction du modele
model = build_model_v2()
total     = model.count_params()
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'Modele : {model.name}')
print(f'  Parametres total        : {total:,}')
print(f'  Parametres entrainables : {trainable:,}')
print(f'  Parametres geles        : {total - trainable:,}')

print('\nTete de classification :')
for layer in model.layers[-12:]:
    try:
        shape = str(layer.output.shape)
    except:
        shape = 'shape non disponible'
    print(f'  {layer.name:<35} {shape}')

print('\nModele construit avec succes !')
print('Innovations ajoutees :')
print('  1. CBAM     Channel Attention + Spatial Attention')
print('  2. Multi-Scale Fusion    3 niveaux de features fusionnes')
print('  3. Blocs Residuels    skip connections dans la tete')

In [ ]:
# # =============================================================
# # CELLULE 5    Diagramme visuel de l'architecture
# # =============================================================

# !pip install visualkeras --quiet

# import visualkeras
# from PIL import ImageFont
# import matplotlib.pyplot as plt

# print('GÉNÉRATION DU DIAGRAMME VISUEL (VUE 3D)...')
# print('='*60)

# try:
#     img = visualkeras.layered_view(
#         model, 
#         legend=True, 
#         spacing=20, 
#         draw_volume=True,
#         max_xy=400,
#         min_z=10,
#         max_z=100
#     )

#     plt.figure(figsize=(15, 12))
#     plt.imshow(img)
#     plt.axis('off')
#     plt.title('Architecture DenseNet121 + CBAM + Multi-Scale + Résiduel', fontsize=14, pad=20)
    
#     img.save(f'{WORK_PATH}/logs/architecture_3D.png')
#     plt.show()
    
#     print(f'Diagramme sauvegardé avec succès dans logs/architecture_3D.png')

# except Exception as e:
#     print(f"Erreur lors de la génération visuelle : {e}")
#     print("\nAlternative : Résumé détaillé des connexions de la tête :")
#     model.summary()

In [ ]:
# # =============================================================
# # CELLULE 5555555    Diagramme visuel de l'architecture (Correction Kaggle)
# # =============================================================

# import os
# import tensorflow as tf
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg

# print('GÉNÉRATION DE L\'IMAGE PNG DE L\'ARCHITECTURE...')
# print('='*60)

# # Pour éviter le crash de Graphviz sur Kaggle avec les gros modèles,
# # on désactive l'affichage des formes internes complexes (show_shapes=False)
# # et on force une taille maximale. C'est le seul moyen d'obtenir le PNG.
# from tensorflow.keras.utils import plot_model

# img_path = 'architecture_modele.png'

# try:
#     # Génération du fichier PNG
#     plot_model(
#         model,
#         to_file=img_path,
#         show_shapes=False,         # CRUCIAL sur Kaggle : évite le crash mémoire
#         show_layer_names=True,     # Permet de voir vos blocs CBAM et Fusion
#         rankdir='TB',              # Affichage vertical de haut en bas
#         dpi=70                     # Résolution adaptée pour ne pas saturer Graphviz
#     )

#     # Lecture et affichage immédiat de l'image PNG générée
#     if os.path.exists(img_path):
#         img = mpimg.imread(img_path)
        
#         # On ajuste la taille de la figure pour que ce soit grand et lisible
#         plt.figure(figsize=(12, 30)) 
#         plt.imshow(img)
#         plt.axis('off')
#         plt.title('Architecture Globale : DenseNet121 + CBAM + Multi-Scale', fontsize=14, pad=20)
#         plt.tight_layout()
#         plt.show()
#         print(f"L'image a été générée avec succès et sauvegardée sous : {img_path}")
#     else:
#         print("Le fichier PNG n'a pas pu être créé.")

# except Exception as e:
#     print(f"\nGraphviz a encore bloqué. Tentative avec la méthode alternative par blocs...")
    
#     # Si même sans les shapes ça plante, on utilise Visualkeras qui génère un PNG à 100%
#     !pip install visualkeras --quiet
#     import visualkeras
    
#     img_blocks = visualkeras.layered_view(model, legend=True, spacing=15, draw_volume=True)
#     img_blocks.save(img_path)
    
#     img = mpimg.imread(img_path)
#     plt.figure(figsize=(15, 12))
#     plt.imshow(img)
#     plt.axis('off')
#     plt.show()
#     print(f"Image PNG alternative (vue en blocs) sauvegardée sous : {img_path}")

In [ ]:
# ============================================================
# CELLULE E 6    Entraînement complet (Phase 1 + Phase 2) 
# ============================================================


# Class weights
print('Calcul des class weights corriges...')
class_weights = {}
n = len(df_train_s)
for i, label in enumerate(ALL_LABELS):
    n_pos = df_train_s[label].sum()
    if n_pos == 0:
        class_weights[i] = 1.0
        continue
    ratio = (n - n_pos) / n_pos
    class_weights[i] = min(ratio / N_CLASSES, 10.0)

print(f'{"Pathologie":<25} {"Poids":>8}  {"Nb images":>10}')
print('-'*50)
for i, label in enumerate(ALL_LABELS):
    n_pos = df_train_s[label].sum()
    print(f'{label:<25} {class_weights[i]:>8.3f}  {n_pos:>10,}')

# un sous-ensemble
sampled = []
for label in ALL_LABELS:
    limit  = 8000 if label == 'No Finding' else 5000
    subset = df_train[df_train[label] == 1].head(limit)
    sampled.append(subset)
df_train_s = pd.concat(sampled).drop_duplicates(subset='Image Index')
df_train_s = df_train_s.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nSous-ensemble ameliore : {len(df_train_s):,} images (avant: ~29000)')

# generateurs avec sous-ensemble
train_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train_s, x_col='FilePath', y_col=ALL_LABELS,
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='raw', color_mode='rgb', shuffle=True, seed=42
)
print(f'Steps/epoch : {len(train_gen)}')

# PHASE 1 : base gelee
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        AUC(name='auc', multi_label=True),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

cb_p1 = [
    ModelCheckpoint(f'{WORK_PATH}/models/best_p1.keras',
                    monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=6,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.3,
                      patience=3, min_lr=1e-7, verbose=1),
    CSVLogger(f'{WORK_PATH}/logs/phase1.csv')
]

print('\nPHASE 1    Entrainement tete (base gelee, 20 epochs max)')
history_p1 = model.fit(
    train_gen, epochs=20, validation_data=val_gen,
    callbacks=cb_p1, class_weight=class_weights, verbose=1
)
best_p1_auc = max(history_p1.history['val_auc'])
print(f'Phase 1 terminee    val_auc : {best_p1_auc:.4f}')

# PHASE 2 : Fine-tuning
CUSTOM_OBJECTS = {'ChannelMeanPool': ChannelMeanPool, 'ChannelMaxPool': ChannelMaxPool}
model = tf.keras.models.load_model(
    f'{WORK_PATH}/models/best_p1.keras',
    custom_objects=CUSTOM_OBJECTS
)

# degele 200 couches
TETE_KEYWORDS = ['gap','dense','bn','relu','dropout','projection',
                 'residual','prediction','fusion','add','multiply',
                 'concatenate','activation','channel','spatial','reshape']
TETE_NAMES = {l.name for l in model.layers[-20:]
              if any(k in l.name.lower() for k in TETE_KEYWORDS)}
for layer in model.layers:
    layer.trainable = False
base_layers = [l for l in model.layers if l.name not in TETE_NAMES]
for layer in base_layers[-200:]:
    layer.trainable = True
for layer in model.layers:
    if layer.name in TETE_NAMES:
        layer.trainable = True

n_train = sum(1 for l in model.layers if l.trainable)
print(f'\nCouches entrainables : {n_train}/{len(model.layers)} ({n_train/len(model.layers)*100:.1f}%)')

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        AUC(name='auc', multi_label=True),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

cb_p2 = [
    ModelCheckpoint(f'{WORK_PATH}/models/best_p2.keras',
                    monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5,
                      patience=4, min_lr=1e-8, verbose=1),
    CSVLogger(f'{WORK_PATH}/logs/phase2.csv')
]

print('\nPHASE 2    Fine-tuning (200 couches, lr=1e-5, 30 epochs max)')
history_p2 = model.fit(
    train_gen, epochs=30, validation_data=val_gen,
    callbacks=cb_p2, class_weight=class_weights, verbose=1
)
best_p2_auc = max(history_p2.history['val_auc'])
print(f'\nPhase 2 terminee    val_auc : {best_p2_auc:.4f}')
print(f'Amelioration           : +{best_p2_auc - best_p1_auc:.4f}')

In [ ]:
# import json

# ALL_LABELS = [
#     'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
#     'Effusion', 'Emphysema', 'Fibrosis', 'Hernia',
#     'Infiltration', 'Mass', 'No Finding', 'Nodule',
#     'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
# ]

# with open(f'{WORK_PATH}/models/labels.json', 'w') as f:
#     json.dump(ALL_LABELS, f)
# print('labels.json cree')

# metrics_export = {
#     'mean_auc'        : 0.7455,
#     'f1_macro'        : 0.15,
#     'f1_micro'        : 0.28,
#     'accuracy'        : 0.33,
#     'precision_macro' : 0.37,
#     'recall_macro'    : 0.06,
#     'img_size'        : 224,
#     'n_classes'       : 15,
#     'labels'          : ALL_LABELS,
#     'model_name'      : 'DenseNet121_CBAM_MultiScale',
#     'architecture'    : 'DenseNet121 + CBAM + Multi-Scale + Residual Blocks',
#     'auc_per_class'   : {l: 0.74 for l in ALL_LABELS}
# }
# with open(f'{WORK_PATH}/models/metrics.json', 'w') as f:
#     json.dump(metrics_export, f, indent=2)
# print('metrics.json cree')
# print('Telecharge ces 2 fichiers depuis Output maintenant !')

In [ ]:
# # =============================================================
# # CELLULE 7    Courbes d'apprentissage + Diagnostic
# # =============================================================

# auc_t  = history_p1.history['auc']          + history_p2.history['auc']
# auc_v  = history_p1.history['val_auc']      + history_p2.history['val_auc']
# loss_t = history_p1.history['loss']         + history_p2.history['loss']
# loss_v = history_p1.history['val_loss']     + history_p2.history['val_loss']
# acc_t  = history_p1.history['accuracy']     + history_p2.history['accuracy']
# acc_v  = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
# ep1    = len(history_p1.history['loss'])
# epochs_range = range(1, len(auc_t)+1)

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# fig.suptitle('Courbes apprentissage    Phase 1 + Fine-tuning', fontsize=13, fontweight='bold')

# configs = [
#     (auc_t,  auc_v,  'AUC Multi-label',         'AUC',      [0.5, 1.01]),
#     (loss_t, loss_v, 'Binary Crossentropy Loss', 'Loss',     None),
#     (acc_t,  acc_v,  'Accuracy',                 'Accuracy', [0, 1.01]),
# ]
# for ax, (tr, vl, title, ylabel, ylim) in zip(axes, configs):
#     ax.plot(epochs_range, tr, label='Train', color='#2196F3', linewidth=2)
#     ax.plot(epochs_range, vl, label='Val',   color='#FF5722', linewidth=2)
#     ax.axvline(x=ep1, color='gray', linestyle='--', linewidth=1.5,
#                label=f'Fine-tuning epoch {ep1+1}')
#     ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
#     ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
#     if ylim: ax.set_ylim(ylim)

# plt.tight_layout()
# plt.savefig(f'{WORK_PATH}/logs/learning_curves.png', dpi=120)
# plt.show()

# gap       = max(auc_t[-5:]) - max(auc_v[-5:])
# final_auc = max(auc_v)
# print('DIAGNOSTIC')
# print('='*45)
# print(f'  val_auc final   : {final_auc:.4f}')
# print(f'  train_auc final : {max(auc_t[-5:]):.4f}')
# print(f'  Ecart train/val : {gap:.4f}')
# if final_auc >= 0.80 and gap <= 0.05:
#     print('EXCELLENT')
# elif final_auc >= 0.72 and gap <= 0.08:
#     print('BON')
# elif gap > 0.10:
#     print('OVERFITTING')
# else:
#     print('UNDERFITTING')


In [ ]:
# =============================================================
# CELLULE 7    Récupération des Courbes depuis les Logs (Kaggle)
# =============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

try:
    log_dir = f"{WORK_PATH}/logs"
except NameError:
    log_dir = "/kaggle/working/logs"

p1_path = os.path.join(log_dir, "phase1.csv")
p2_path = os.path.join(log_dir, "phase2.csv")

print("Récupération de l'historique d'entraînement depuis les fichiers CSV...")

try:
    df_p1 = pd.read_csv(p1_path)
    df_p2 = pd.read_csv(p2_path)
    
    auc_t  = list(df_p1['auc'])          + list(df_p2['auc'])
    auc_v  = list(df_p1['val_auc'])      + list(df_p2['val_auc'])
    loss_t = list(df_p1['loss'])         + list(df_p2['loss'])
    loss_v = list(df_p1['val_loss'])     + list(df_p2['val_loss'])
    acc_t  = list(df_p1['accuracy'])     + list(df_p2['accuracy'])
    acc_v  = list(df_p1['val_accuracy']) + list(df_p2['val_accuracy'])
    
    ep1 = len(df_p1)
    epochs_range = range(1, len(auc_t) + 1)
    
    # graphiques Matplotlib
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Courbes apprentissage    Phase 1 + Fine-tuning (Restaurées via CSV)', fontsize=13, fontweight='bold')

    configs = [
        (auc_t,  auc_v,  'AUC Multi-label',         'AUC',      [0.5, 1.01]),
        (loss_t, loss_v, 'Binary Crossentropy Loss', 'Loss',     None),
        (acc_t,  acc_v,  'Accuracy',                 'Accuracy', [0, 1.01]),
    ]
    
    for ax, (tr, vl, title, ylabel, ylim) in zip(axes, configs):
        ax.plot(epochs_range, tr, label='Train', color='#2196F3', linewidth=2)
        ax.plot(epochs_range, vl, label='Val',   color='#FF5722', linewidth=2)
        ax.axvline(x=ep1, color='gray', linestyle='--', linewidth=1.5,
                   label=f'Fine-tuning epoch {ep1+1}')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        if ylim: ax.set_ylim(ylim)

    plt.tight_layout()
    plt.savefig(f'{log_dir}/learning_curves.png', dpi=120)
    plt.show()

    # calcul du bloc Diagnostic final
    gap       = max(auc_t[-5:]) - max(auc_v[-5:])
    final_auc = max(auc_v)
    
    print('\nDIAGNOSTIC AUTOMATIQUE')
    print('='*45)
    print(f'  val_auc final   : {final_auc:.4f}')
    print(f'  train_auc final : {max(auc_t[-5:]):.4f}')
    print(f'  Ecart train/val : {gap:.4f}')
    
    if final_auc >= 0.80 and gap <= 0.05:
        print('EXCELLENT')
    elif final_auc >= 0.72 and gap <= 0.08:
        print('BON')
    elif gap > 0.10:
        print('OVERFITTING')
    else:
        print('UNDERFITTING')

except FileNotFoundError:
    print("Erreur : Les fichiers de logs n'ont pas été trouvés.")
    print(f"Vérifiez la présence de 'phase1.csv' et 'phase2.csv' dans le dossier : {log_dir}")

In [ ]:
# =============================================================
# CELLULE 8 F    CELLULE F    Metriques completes 
#  (Accuracy, Precision, Recall, F1, AUC)
# =============================================================
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report,
    accuracy_score
)

CUSTOM_OBJECTS = {'ChannelMeanPool': ChannelMeanPool, 'ChannelMaxPool': ChannelMaxPool}
model = tf.keras.models.load_model(
    f'{WORK_PATH}/models/best_p2.keras',
    custom_objects=CUSTOM_OBJECTS
)

print('Modele charge    predictions en cours...')
y_pred_proba = model.predict(test_gen, verbose=1)
y_true       = df_test[ALL_LABELS].values
y_pred_bin   = (y_pred_proba > 0.5).astype(int)

# AUC par classe  
print('\nAUC PAR PATHOLOGIE')
print('='*55)
auc_per_class = []
for i, label in enumerate(ALL_LABELS):
    if y_true[:, i].sum() > 0:
        val = roc_auc_score(y_true[:, i], y_pred_proba[:, i])
        auc_per_class.append(val)
        bar = '|' * int(val * 25)
        print(f'  {label:<25} : {val:.4f}  {bar}')
    else:
        auc_per_class.append(None)
        print(f'  {label:<25} : N/A')

valid_aucs = [a for a in auc_per_class if a is not None]
mean_auc   = np.mean(valid_aucs)

# metriques globales  
f1_macro   = f1_score(y_true, y_pred_bin, average='macro',  zero_division=0)
f1_micro   = f1_score(y_true, y_pred_bin, average='micro',  zero_division=0)
f1_weighted= f1_score(y_true, y_pred_bin, average='weighted', zero_division=0)
prec_macro = precision_score(y_true, y_pred_bin, average='macro', zero_division=0)
rec_macro  = recall_score(y_true, y_pred_bin, average='macro', zero_division=0)
acc_global = accuracy_score(y_true.flatten(), y_pred_bin.flatten())

print(f'\nMETRIQUES GLOBALES    TEST SET')
print('='*45)
print(f'  Mean AUC-ROC       : {mean_auc:.4f}')
print(f'  Accuracy globale   : {acc_global:.4f}  ({acc_global*100:.2f}%)')
print(f'  F1-score macro     : {f1_macro:.4f}')
print(f'  F1-score micro     : {f1_micro:.4f}')
print(f'  F1-score weighted  : {f1_weighted:.4f}')
print(f'  Precision macro    : {prec_macro:.4f}')
print(f'  Recall macro       : {rec_macro:.4f}')

# Tableau par pathologie  
print(f'\nTABLEAU COMPLET PAR PATHOLOGIE')
print(f'{"Pathologie":<25} {"AUC":>6}  {"F1":>6}  {"Precision":>9}  {"Rappel":>8}  {"Accuracy":>9}')
print('-'*70)
for i, label in enumerate(ALL_LABELS):
    if y_true[:, i].sum() == 0:
        continue
    auc_v  = auc_per_class[i] if auc_per_class[i] else 0
    f1_v   = f1_score(y_true[:,i], y_pred_bin[:,i], zero_division=0)
    pre_v  = precision_score(y_true[:,i], y_pred_bin[:,i], zero_division=0)
    rec_v  = recall_score(y_true[:,i], y_pred_bin[:,i], zero_division=0)
    acc_v  = accuracy_score(y_true[:,i], y_pred_bin[:,i])
    print(f'{label:<25} {auc_v:>6.3f}  {f1_v:>6.3f}  {pre_v:>9.3f}  {rec_v:>8.3f}  {acc_v:>9.3f}')
print('-'*70)
print(f'{"MOYENNE":<25} {mean_auc:>6.3f}  {f1_macro:>6.3f}  {prec_macro:>9.3f}  {rec_macro:>8.3f}')

# Rapport 
print(f'\nRAPPORT DE CLASSIFICATION SKLEARN')
print('='*60)
print(classification_report(y_true, y_pred_bin,
                             target_names=ALL_LABELS, zero_division=0))

In [ ]:
# ============================================================
# CELLULE G    Visualisations completes des metriques
# ============================================================

# barplt AUC par classe  
valid_labels = [ALL_LABELS[i] for i, a in enumerate(auc_per_class) if a is not None]
valid_vals   = [a for a in auc_per_class if a is not None]
colors_bar   = ['#4CAF50' if v>=0.80 else '#FF9800' if v>=0.70 else '#F44336'
                for v in valid_vals]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(valid_labels, valid_vals, color=colors_bar, edgecolor='white')
ax.axhline(y=mean_auc, color='navy', linestyle='--', linewidth=2,
           label=f'Mean AUC = {mean_auc:.4f}')
ax.axhline(y=0.80, color='green', linestyle=':', linewidth=1.5, label='Objectif 0.80')
ax.axhline(y=0.70, color='orange', linestyle=':', linewidth=1, label='Seuil acceptable 0.70')
ax.set_ylim([0.5, 1.0]); ax.set_ylabel('AUC-ROC')
ax.set_title('AUC-ROC par pathologie    Test Set', fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9); ax.legend()
for bar, val in zip(bars, valid_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/auc_per_class.png', dpi=150)
plt.show()

# F1 / Precision / Recall  
f1_vals   = [f1_score(y_true[:,i], y_pred_bin[:,i], zero_division=0) for i in range(N_CLASSES)]
prec_vals = [precision_score(y_true[:,i], y_pred_bin[:,i], zero_division=0) for i in range(N_CLASSES)]
rec_vals  = [recall_score(y_true[:,i], y_pred_bin[:,i], zero_division=0) for i in range(N_CLASSES)]

x = np.arange(N_CLASSES)
w = 0.28
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x-w, f1_vals,   width=w, label='F1-score',  color='#2196F3', alpha=0.85)
ax.bar(x,   prec_vals, width=w, label='Precision', color='#4CAF50', alpha=0.85)
ax.bar(x+w, rec_vals,  width=w, label='Recall',    color='#FF9800', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ALL_LABELS, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim([0, 1.0])
ax.set_title('F1-score, Precision et Recall par pathologie    Test Set', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/f1_precision_recall.png', dpi=150)
plt.show()

# Courbes ROC Top 5  
top5    = sorted(range(len(ALL_LABELS)),
                 key=lambda i: auc_per_class[i] if auc_per_class[i] else 0, reverse=True)[:5]
palette = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800']
fig, ax = plt.subplots(figsize=(8, 7))
for idx, color in zip(top5, palette):
    fpr, tpr, _ = roc_curve(y_true[:, idx], y_pred_proba[:, idx])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{ALL_LABELS[idx]} (AUC={auc_per_class[idx]:.3f})')
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Aleatoire (AUC=0.5)')
ax.fill_between([0,1],[0,1],[1,1], alpha=0.05, color='green', label='Zone ideale')
ax.set_xlabel('Taux faux positifs (1-Specificite)')
ax.set_ylabel('Taux vrais positifs (Sensibilite)')
ax.set_title('Courbes ROC    Top 5 pathologies', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/roc_curves.png', dpi=150)
plt.show()

# Matrice de confusion Top 3  
top3_labels = [ALL_LABELS[i] for i in top5[:3]]
fig, axes   = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Matrices de Confusion    Top 3 pathologies', fontweight='bold')
for ax, label in zip(axes, top3_labels):
    idx = ALL_LABELS.index(label)
    cm  = confusion_matrix(y_true[:, idx], y_pred_bin[:, idx])
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Negatif','Positif'])
    ax.set_yticklabels(['Negatif','Positif'])
    ax.set_xlabel('Predit'); ax.set_ylabel('Reel')
    ax.set_title(label, fontweight='bold')
    tn, fp, fn, tp = cm.ravel()
    ax.text(0, 0, f'TN\n{tn:,}', ha='center', va='center',
            color='black', fontsize=12, fontweight='bold')
    ax.text(1, 0, f'FP\n{fp:,}', ha='center', va='center',
            color='white' if fp > cm.max()/2 else 'black', fontsize=12, fontweight='bold')
    ax.text(0, 1, f'FN\n{fn:,}', ha='center', va='center',
            color='white' if fn > cm.max()/2 else 'black', fontsize=12, fontweight='bold')
    ax.text(1, 1, f'TP\n{tp:,}', ha='center', va='center',
            color='white' if tp > cm.max()/2 else 'black', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/confusion_matrices.png', dpi=150)
plt.show()

# Sauvegarde finale  
model.save(f'{WORK_PATH}/models/model_final.keras')
with open(f'{WORK_PATH}/models/labels.json', 'w') as f:
    json.dump(ALL_LABELS, f)
metrics_export = {
    'mean_auc': float(mean_auc), 'f1_macro': float(f1_macro),
    'f1_micro': float(f1_micro), 'accuracy': float(acc_global),
    'precision_macro': float(prec_macro), 'recall_macro': float(rec_macro),
    'img_size': IMG_SIZE, 'n_classes': N_CLASSES, 'labels': ALL_LABELS,
    'model_name': 'DenseNet121_CBAM_MultiScale',
    'architecture': 'DenseNet121 + CBAM + Multi-Scale + Residual Blocks',
    'auc_per_class': {
        label: float(auc_per_class[i])
        for i, label in enumerate(ALL_LABELS) if auc_per_class[i] is not None
    }
}
with open(f'{WORK_PATH}/models/metrics.json', 'w') as f:
    json.dump(metrics_export, f, indent=2)
print(f'Fichiers sauvegardes    Mean AUC={mean_auc:.4f} | Accuracy={acc_global:.4f}')